In [1]:
# dynamo is designed to
# estimate direction and magnitude of expression dynamics (RNA velocity),
# predict the entire lineage trajectory of any intial cell state (vector field),
# characterize the structure (vector field topology) of full gene expression space, and fate commitment potential (single cell potential).

In [2]:
pip install harmonypy

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install louvain

^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


In [4]:
#pip install kdepy
import loompy
import scvelo as scv
import scanpy as sc

In [5]:
import numpy as np

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation

from IPython.display import display, HTML

import dynamo as dyn
dyn.configuration.set_figure_params('dynamo', background='white')
dyn.get_all_dependencies_version()

ModuleNotFoundError: No module named 'dynamo'

In [ ]:
adata_10 = dyn.read('normal-CIS_integrated_dynamo.h5ad')

In [ ]:
sc.pl.umap(adata_10, color='louvain', legend_loc='right margin' )

In [ ]:
scv.pl.umap(adata_10, color='louvain', legend_loc='on data')

In [ ]:
scv.pl.scatter(adata_10, groups=[[c] for c in adata_10.obs['batch'].cat.categories], color='batch', ncols=4, save="scatterplot_cluster")

In [ ]:
adata_10.obs['louvain'].dtype

In [ ]:
##############################################
############## Subclustering #################
###############################################

In [ ]:
adata_11 = adata_10[adata_10.obs['louvain'].isin(['Epi_2', 'Epi_3','Epi_4','New_1', 'New_2', 'New_3', 'New_4',
                  'New_5', 'New_6', 'New_7', 'New_8', 'New_9', 'New_10',
                  'New_11',])]

In [ ]:
scv.pl.umap(adata_11, color='louvain', legend_loc='on data')

In [ ]:
scv.pl.scatter(adata_11, groups=[[c] for c in adata_11.obs['batch'].cat.categories], color='batch', ncols=4, save="scatterplot_subcluster")

In [ ]:
# 2. Preprocessing
dyn.pp.recipe_monocle(adata_11)
# the author use dyn.pp.recipe_monocle(adata), which uses similar but generalized strategy from Monocle 3:
# converting ensemble gene names to gene official name and set them as .var_names if needed.
# making cell (.obs_names) and gene names (.var_names) unique.
# detecting your experiment type (conventional scRNA-seq or time-resolved metabolic labeling datasets) and normalize layers (the spliced and unspliced or metabolic labelled, and total mRNAs) and log1p transform the data
# calculating number of expressed genes (nGenes), total expression values (nCounts), percentage of total mitochondria gene values (pMito) for each cell and save them to .obs.
# calculating each cell’s cell cycle stage score.
# calculating new to total ratio (ntr) for each gene and cell.
# normalizes all datasets in different layers, do feature selection, log1p transformation and PCA dimension reduction.
# adding collapsed new, total and unspliced, spliced layers from the uu, ul, su, sl layers of a metabolic labeling experiment.
adata_11

In [ ]:
# 3. RNA velocity with parallelism
# 3 modes: deterministic, stochastic and negbin. Negbin is a novel method that relies on the negative binomial formulation of gene exrpession distribution at steady state.
dyn.tl.dynamics(adata_11, model='stochastic', cores=40)
# dyn.tl.dynamics() checks the experimental type automatically and learn the velocity for each feature gene, usually stochastic or negative binomial method for conventional scRNA-seq; kinetic, degradation or one-shot models for metabolic labeling based scRNA-seq.
adata_11

In [ ]:
dyn.tl.reduceDimension(adata_11, basis='umap', n_pca_components=30, n_neighbors=30, reduction_method='umap', cores=40)
# if basis='pca', it uses HDBSCAN for clustering, you can use basis='umap'
adata_11

In [ ]:
adata_11.obs['louvain'].dtype

In [ ]:
dyn.pl.umap(adata_11, color='louvain')

In [ ]:
# if you observe undesired backward vector flow. You can evaluate the confidence of gene-wise velocity via
    ## dyn.tl.gene_wise_confidence(adata, group='group', lineage_dict={'Progenitor': ['terminal_cell_state']})
    # lineage_dict is a dictionary indicates the progenitor group, the values is a list including the possible terminal cell groups

In [ ]:
# check phase diagram
dyn.pl.phase_portraits(adata_11, genes=adata.var_names[adata_11.var.use_for_dynamics][:4], basis='umap', color='louvain')
# pvalb1 gene has a strange phase diagram with a few cells have high spliced expression values but extremely low unspliced expression values.
# This kind of phase space may points to inproper intron capture of those genes during the library prepartion or sequencing and they should never be used for velocity projection and vector field analysis.

In [ ]:
# 4. Velocity projection
# The projection calculates a transition matrix between cells for local averaging of velocity vectors into low dimension.
dyn.tl.cell_velocities(adata_11, basis='umap', n_neighbors = 30, method='pearson', other_kernels_dict={'transform': 'sqrt'})
# There are 3 methods to calculate the transition matrix, either kmc, cosine, pearson. kmc is our new approach to learn the transition matrix via diffusion approximation or an Itô kernel. cosine or pearson are the methods used in the original velocyto or the scvelo paper.
adata_11

In [ ]:
# check the confidence of cell-wise velocity to understand how reliable the recovered velocity is across cells
dyn.tl.cell_wise_confidence(adata_11, method='jaccard')
adata_11

In [ ]:
adata_11.obs['louvain']

In [ ]:
## dyn.tl.confident_cell_velocities(adata, group='Cell_type', lineage_dict={'Proliferating Progenitor': ['Schwann Cell', 'Melanophore']}, basis='umap', confidence_threshold=0.85)
adata_11

In [ ]:
dyn.pl.cell_wise_vectors(adata_11, basis='umap', color=['louvain'], show_legend='on data', quiver_length=6, quiver_size=6, pointsize=0.1, show_arrowed_spines=False)

In [ ]:
#previous image
#dyn.pl.cell_wise_vectors(adata_11, basis='umap', color=['louvain'], show_legend='on data', quiver_length=6, quiver_size=6, pointsize=0.1, show_arrowed_spines=False)

In [ ]:
dyn.pl.streamline_plot(adata_11, color=['louvain','batch'], basis='umap', show_legend='on data', show_arrowed_spines=True, pointsize=0.3, theme='fire')

In [ ]:
#previous image
#dyn.pl.streamline_plot(adata_11, color=['louvain','batch'], basis='umap', show_legend='on data', show_arrowed_spines=True, pointsize=0.3, theme='fire')

In [ ]:
dyn.pl.streamline_plot(adata_11, color=['louvain'], basis='umap', show_legend=True, show_arrowed_spines=True, pointsize=0.2, theme='fire',
                       ax=None, sort='raw', aggregate=None, inverse=False, cell_inds='all', method='gaussian', 
                       xy_grid_nums=[50, 50], cut_off_velocity=True, density=1, linewidth=1, streamline_alpha=1, vector='velocity',
                       frontier=False, save_show_or_return='save', 
                       save_kwargs={"path":'C:/Users/KKo1/OneDrive - mdanderson.org/13. KyungPil Ko/Experiment_results/scRNA-seq/2020_nature_comm/Dynamo/Integrated/subcluster',
                                   "prefix": "streamline_legend_", "dpi": None, "ext": "svg", "transparent": True}, 
                       s_kwargs_dict={})

In [ ]:
dyn.pl.streamline_plot(adata_11, color=['louvain'], basis='umap', show_legend='on data', show_arrowed_spines=True, pointsize=0.2, theme='fire',
                       ax=None, sort='raw', aggregate=None, inverse=False, cell_inds='all', method='gaussian', 
                       xy_grid_nums=[50, 50], cut_off_velocity=True, density=1, linewidth=1, streamline_alpha=1, vector='velocity',
                       frontier=False, save_show_or_return='save', 
                       save_kwargs={"path":'C:/Users/KKo1/OneDrive - mdanderson.org/13. KyungPil Ko/Experiment_results/scRNA-seq/2020_nature_comm/Dynamo/Integrated/subcluster',
                                   "prefix": "streamline_", "dpi": None, "ext": "svg", "transparent": True}, 
                       s_kwargs_dict={})

In [ ]:
# 5. Reconstruct vector field
dyn.vf.VectorField(adata_11, basis='umap', M=1000, pot_curl_div=True)
adata_11

In [ ]:
dyn.pl.plot_energy(adata_11, basis='umap')
adata_11

In [ ]:
# 6. Characterize vector field topology
# characterize the topology of the full vector field
dyn.pl.topography(adata_11, basis='umap', color=['ntr', 'Clusters'], background='white', streamline_color='black', show_legend='on data', frontier=True)
# the fixed points corresponds to terminal cell types or progenitors: black is absorbing fixed points; red is emitting fixed points; blue is unstable fixed points.
adata_11

In [ ]:
# like pseudotime
dyn.pl.umap(adata_11, color='umap_ddhodge_potential', frontier=True)

In [ ]:
dyn.pl.scatters(adata_11, x=np.repeat('umap_ddhodge_potential', 4), y=['Mki67','Trp63','Sox2','Krt13'],
                color='louvain', layer='X_spliced', background='white', ncols=3, pointsize=0.25, alpha=0.8)

In [ ]:
adata_11

In [ ]:
dyn.cleanup(adata_11)
del adata_11.obs['control_point_pca']
del adata_11.uns['velocyto_SVR']
del adata_11.uns['VecFld_umap']
#del adata_11.uns['fate_umap']
adata_11

In [ ]:
adata_11.write('normal-CIS_integrated_subcluster_dynamo.h5ad')

In [ ]:
###############################################################
################ subcluster by Batch ##########################
###############################################################

In [ ]:
import os
os.chdir('C:/Users/KKo1/OneDrive - mdanderson.org/13. KyungPil Ko/Experiment_results/scRNA-seq/2020_nature_comm/Dynamo/Integrated')
adata_11 = dyn.read('normal-CIS_integrated_subcluster_dynamo.h5ad')

In [ ]:
adata_11.obs['batch'].dtype

In [ ]:
adata_N = adata_11[adata_11.obs['batch'].isin(['Normal'])]
adata_I = adata_11[adata_11.obs['batch'].isin(['Inflammation'])]
adata_H = adata_11[adata_11.obs['batch'].isin(['Hyperplasia'])]
adata_D = adata_11[adata_11.obs['batch'].isin(['Dysplasia'])]
adata_C = adata_11[adata_11.obs['batch'].isin(['CIS'])]

In [ ]:
dyn.pl.streamline_plot(adata_N, color=['louvain'], basis='umap', show_legend='on data', show_arrowed_spines=True, pointsize=0.3, theme='fire',
                       ax=None, sort='raw', aggregate=None, inverse=False, cell_inds='all', method='gaussian', 
                       xy_grid_nums=[50, 50], cut_off_velocity=True, density=1, linewidth=1, streamline_alpha=1, vector='velocity',
                       frontier=False, save_show_or_return='save', 
                       save_kwargs={"path":'C:/Users/KKo1/OneDrive - mdanderson.org/13. KyungPil Ko/Experiment_results/scRNA-seq/2020_nature_comm/Dynamo/Integrated/subcluster/normal',
                                   "prefix": "streamline_", "dpi": None, "ext": "svg", "transparent": True}, 
                       s_kwargs_dict={})

In [ ]:
dyn.pl.streamline_plot(adata_I, color=['louvain'], basis='umap', show_legend='on data', show_arrowed_spines=True, pointsize=0.3, theme='fire',
                      ax=None, sort='raw', aggregate=None, inverse=False, cell_inds='all', method='gaussian', 
                       xy_grid_nums=[50, 50], cut_off_velocity=True, density=1, linewidth=1, streamline_alpha=1, vector='velocity',
                       frontier=False, save_show_or_return='save', 
                       save_kwargs={"path":'C:/Users/KKo1/OneDrive - mdanderson.org/13. KyungPil Ko/Experiment_results/scRNA-seq/2020_nature_comm/Dynamo/Integrated/subcluster/Infla',
                                   "prefix": "streamline_", "dpi": None, "ext": "svg", "transparent": True}, 
                       s_kwargs_dict={})

In [ ]:
dyn.pl.streamline_plot(adata_H, color=['louvain'], basis='umap', show_legend='on data', show_arrowed_spines=True, pointsize=0.3, theme='fire')

In [ ]:
dyn.pl.streamline_plot(adata_D, color=['louvain'], basis='umap', show_legend='on data', show_arrowed_spines=True, pointsize=0.3, theme='fire')

In [ ]:
dyn.pl.streamline_plot(adata_C, color=['louvain'], basis='umap', show_legend='on data', show_arrowed_spines=True, pointsize=0.3, theme='fire')

In [ ]:
######################################################################################
############################ Normal analysis #########################################
######################################################################################

In [ ]:
# 5. Reconstruct vector field
dyn.vf.VectorField(adata_N, basis='umap', M=1000, pot_curl_div=True)
adata_N

In [ ]:
dyn.pl.plot_energy(adata_N, basis='umap')
adata_N

In [ ]:
# 6. Characterize vector field topology
# characterize the topology of the full vector field
dyn.pl.topography(adata_N, basis='umap', color=['ntr', 'Clusters'], background='white', streamline_color='black', show_legend='on data', frontier=True)
# the fixed points corresponds to terminal cell types or progenitors: black is absorbing fixed points; red is emitting fixed points; blue is unstable fixed points.
adata_N

In [ ]:
# 7. Beyond RNA velocity
dyn.tl.cell_velocities(adata_N, basis='pca')
dyn.vf.VectorField(adata_N, basis='pca')
dyn.vf.speed(adata_N, basis='pca')
dyn.vf.curl(adata_N, basis='umap')
dyn.vf.divergence(adata_N, basis='pca')
dyn.vf.acceleration(adata_N, basis='pca')
dyn.vf.curvature(adata_N, basis='pca')

In [ ]:
# 8. Integrative analysis
dyn.pl.cell_wise_vectors(adata_N, color='speed_pca', pointsize=0.5, alpha = 0.7, quiver_length=6, quiver_size=6, save_show_or_return='return')
dyn.pl.grid_vectors(adata_N, color='divergence_pca', quiver_length=12, quiver_size=12, save_show_or_return='return')
dyn.pl.streamline_plot(adata_N, color='acceleration_pca', save_show_or_return='return')
dyn.pl.streamline_plot(adata_N, color='curvature_pca', save_show_or_return='return')

In [ ]:
# 9. Animate fate transition
progenitor = adata_N.obs_names[adata_N.obs.louvain.isin(['Epi_2','New_4','New_6','New_8','New_7'])]    # select some progenitor cells as the initial cell states
len(progenitor)

In [ ]:
# predict the cell fate trajectory via integrating with the vector field function, starting from the initial cell states
dyn.pd.fate(adata_N, init_cells=progenitor, basis='umap', direction='forward', interpolation_num=100, average=False, inverse_transform=False, cores=40)
adata_N

In [ ]:
# use the topography plot as the background, the animated components from each frame will be plotted on its top
fig, ax = plt.subplots()
ax = dyn.pl.topography(adata_N, color='louvain', ax=ax, save_show_or_return='return')

In [ ]:
instance = dyn.mv.StreamFuncAnim(adata_N, color='louvain', ax=ax)
matplotlib.rcParams['animation.embed_limit'] = 2**128    # Ensure all frames will be embedded.
anim = animation.FuncAnimation(instance.fig, instance.update, init_func=instance.init_background, frames=np.arange(100), interval=100, blit=True)
HTML(anim.to_jshtml())    # embedding the movie to jupyter notebook

In [ ]:
# or save the animation as an gif file
fig, ax = plt.subplots()
ax = dyn.pl.topography(adata_N, color='louvain', ax=ax, save_show_or_return='return')
dyn.mv.animate_fates(adata_N, color='louvain', basis='umap', n_steps=100, fig=fig, ax=ax, save_show_or_return='save', logspace=True, max_time=None)

In [ ]:
# use the topography plot as the background, the animated components from each frame will be plotted on its top
fig, ax = plt.subplots()
ax = dyn.pl.topography(adata_N, color='louvain', ax=ax, save_show_or_return='return')

In [ ]:
instance = dyn.mv.StreamFuncAnim(adata_N, color='louvain', ax=ax)
matplotlib.rcParams['animation.embed_limit'] = 2**128    # Ensure all frames will be embedded.
anim = animation.FuncAnimation(instance.fig, instance.update, init_func=instance.init_background, frames=np.arange(100), interval=100, blit=True)
HTML(anim.to_jshtml())    # embedding the movie to jupyter notebook

In [ ]:
# or save the animation as an gif file
fig, ax = plt.subplots()
ax = dyn.pl.topography(adata_N, color='louvain', ax=ax, save_show_or_return='return')
dyn.mv.animate_fates(adata_N, color='louvain', basis='umap', n_steps=100, fig=fig, ax=ax, save_show_or_return='save', logspace=True, max_time=None)

In [ ]:
dyn.cleanup(adata_N)
#del adata_N.uns['velocyto_SVR']
del adata_N.uns['VecFld_umap']
del adata_N.uns['fate_umap']
adata_N

In [ ]:
#del adata_N.uns['batch_colors']

In [ ]:
adata_N.write('normal_subcluster_from_integrated_dynamo.h5ad')

In [ ]:
import os
os.chdir('C:/Users/KKo1/OneDrive - mdanderson.org/13. KyungPil Ko/Experiment_results/scRNA-seq/2020_nature_comm/Dynamo/Integrated')
adata_N = dyn.read('normal_subcluster_from_integrated_dynamo.h5ad')

In [ ]:
######################################################################################
############################ CIS analysis #########################################
######################################################################################

In [ ]:
# 5. Reconstruct vector field
dyn.vf.VectorField(adata_C, basis='umap', M=1000, pot_curl_div=True)
adata_C

In [ ]:
dyn.pl.plot_energy(adata_C, basis='umap')
adata_C

In [ ]:
# 6. Characterize vector field topology
# characterize the topology of the full vector field
dyn.pl.topography(adata_C, basis='umap', color=['ntr', 'Clusters'], background='white', streamline_color='black', show_legend='on data', frontier=True)
# the fixed points corresponds to terminal cell types or progenitors: black is absorbing fixed points; red is emitting fixed points; blue is unstable fixed points.
adata_C

In [ ]:
# 7. Beyond RNA velocity
dyn.tl.cell_velocities(adata_C, basis='pca')
dyn.vf.VectorField(adata_C, basis='pca')
dyn.vf.speed(adata_C, basis='pca')
dyn.vf.curl(adata_C, basis='umap')
dyn.vf.divergence(adata_C, basis='pca')
dyn.vf.acceleration(adata_C, basis='pca')
dyn.vf.curvature(adata_C, basis='pca')

In [ ]:
# 8. Integrative analysis
dyn.pl.cell_wise_vectors(adata_C, color='speed_pca', pointsize=0.5, alpha = 0.7, quiver_length=6, quiver_size=6, save_show_or_return='return')
dyn.pl.grid_vectors(adata_C, color='divergence_pca', quiver_length=12, quiver_size=12, save_show_or_return='return')
dyn.pl.streamline_plot(adata_C, color='acceleration_pca', save_show_or_return='return')
dyn.pl.streamline_plot(adata_C, color='curvature_pca', save_show_or_return='return')

In [ ]:
# 9. Animate fate transition
progenitor = adata_C.obs_names[adata_C.obs.louvain.isin(['Epi_2','New_4','New_6','New_8', 'New_7'])]    # select some progenitor cells as the initial cell states
len(progenitor)

In [ ]:
# predict the cell fate trajectory via integrating with the vector field function, starting from the initial cell states
dyn.pd.fate(adata_C, init_cells=progenitor, basis='umap', direction='forward', interpolation_num=100, average=False, inverse_transform=False, cores=40)
adata_C

In [ ]:
# use the topography plot as the background, the animated components from each frame will be plotted on its top
fig, ax = plt.subplots()
ax = dyn.pl.topography(adata_C, color='louvain', ax=ax, save_show_or_return='return')

In [ ]:
instance = dyn.mv.StreamFuncAnim(adata_C, color='louvain', ax=ax)
matplotlib.rcParams['animation.embed_limit'] = 2**128    # Ensure all frames will be embedded.
anim = animation.FuncAnimation(instance.fig, instance.update, init_func=instance.init_background, frames=np.arange(100), interval=100, blit=True)
HTML(anim.to_jshtml())    # embedding the movie to jupyter notebook

In [ ]:
# or save the animation as an gif file
fig, ax = plt.subplots()
ax = dyn.pl.topography(adata_C, color='louvain', ax=ax, save_show_or_return='return')
dyn.mv.animate_fates(adata_C, color='louvain', basis='umap', n_steps=100, fig=fig, ax=ax, save_show_or_return='save', logspace=True, max_time=None)

In [ ]:
dyn.cleanup(adata_C)
#del adata_C.uns['velocyto_SVR']
del adata_C.uns['VecFld_umap']
del adata_C.uns['fate_umap']
adata_C

In [ ]:
#del adata_C.uns['batch_colors']

In [ ]:
adata_C.write('cis_subcluster_from_integrated_dynamo.h5ad')

In [ ]:
import os
os.chdir('C:/Users/KKo1/OneDrive - mdanderson.org/13. KyungPil Ko/Experiment_results/scRNA-seq/2020_nature_comm/Dynamo/Integrated')
adata_C = dyn.read('cis_subcluster_from_integrated_dynamo.h5ad')